# Arabic Dialect Identification on IADD: lean camera-ready run

One notebook runs every experiment the reviewers asked for, on one Colab machine. This is the trimmed plan: about 5 GPU-hours on a T4 instead of 8.

| Reviewer point | What this notebook does |
|---|---|
| Checkpoint chosen on the test set | A fixed validation split selects the checkpoint. The test split is used once per run. |
| One seed | MARBERTv2 full fine-tuning: **5 seeds**. Comparison models, capped set and partial depths: **3 seeds**. |
| One split | Bootstrap 95% CIs, plus the classical models and MARBERTv2 on **2 extra splits** (Section 7c). |
| No tuning / CV | Classical: 5-fold CV grid search on the training set. MARBERTv2: learning-rate sweep chosen on validation (Section 7a). |
| Downsampling discards data | `capped` training set (up to 12,000 per dialect) with class-weighted **focal loss**. |
| T4 GPU vs. M2 CPU | Everything runs here. Each row records GPU, CPU, RAM, versions and a session id. |
| Resource reporting | GPU memory, batch sizes, CPU/RAM, energy (CodeCarbon), model size, throughput, latency. |
| Other transformers | AraBERT, CAMeLBERT-DA and AraBART through the same pipeline. |
| Per-class reporting | Precision / recall / F1 / support and confusion matrices for every model. |

**How to run**
1. `Runtime > Change runtime type > T4 GPU`.
2. In Section 2 set `SMOKE_TEST = True` and choose `Runtime > Run all` (5 to 10 minutes). This checks that all four models download and train.
3. Set `SMOKE_TEST = False` and `Run all` again. Try to leave the first 1.5 hours uninterrupted (classical models, learning-rate sweep, seed-42 block).
4. If Colab disconnects or the GPU is taken away, reconnect and `Run all`. Finished runs are skipped.
5. Section 8 builds every table and figure into `results_lean/tables` and `results_lean/figures` on your Drive.

Order of the long runs, most important first: learning-rate sweep, seed-42 block, MARBERTv2 seeds, other transformers, capped focal, extra splits, extra depth seeds.

## 1. Runtime, Drive, packages

In [ ]:
import subprocess, shutil
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout if shutil.which('nvidia-smi')
      else 'No GPU found. Runtime > Change runtime type > T4 GPU')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
!pip install -q codecarbon sentencepiece

In [ ]:
import os, re, gc, json, math, time, uuid, shutil, hashlib, platform, warnings, urllib.request
from contextlib import nullcontext

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
import sklearn
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
transformers.logging.set_verbosity_error()
CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if CUDA else 'cpu')
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| sklearn', sklearn.__version__, '| device', DEVICE)

## 2. Configuration
Everything you might want to change is in this cell.

In [ ]:
SMOKE_TEST = False          # True = tiny subsets, 1 epoch, 1 seed, separate output folder. Run this first.

# ---- paths (Drive) ----
PROJECT_DIR = '/content/drive/MyDrive/Dialect-Classification-Arabic'
IADD_PATH   = f'{PROJECT_DIR}/data/IADD.json'
IADD_URL    = 'https://raw.githubusercontent.com/JihadZa/IADD/main/IADD.json'
SPLIT_DIR   = f'{PROJECT_DIR}/split'                 # fixed split, shared by every run
OUT_DIR     = f'{PROJECT_DIR}/results_lean' + ('_smoke' if SMOKE_TEST else '')
CKPT_DIR    = '/content/checkpoints'                 # local disk, never Drive (checkpoints are ~650 MB)

# ---- data / split ----
LABELS     = ['EGY', 'GLF', 'LEV', 'MGH']            # Iraqi (216 samples) and 'general' excluded
LABEL2ID   = {l: i for i, l in enumerate(LABELS)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}
SPLIT_SEED = 42      # split AND training-set sampling; fixed so seeds only vary the model
TEST_FRAC  = 0.20    # per-class test size = 20% of the smallest class
VAL_FRAC   = 0.10    # per-class val size  = 10% of what is left of the smallest class
CAP_PER_CLASS = 12000  # 'capped' strategy: keep up to this many training samples per class
EXTRA_SPLIT_SEEDS = [7, 123]   # Section 7c (Reviewer A: one split). [] skips it.

# ---- seeds ----
SEEDS       = [42, 43, 44, 45, 46]   # MARBERTv2 full fine-tuning (headline model): all five
SEEDS_SHORT = SEEDS[:3]              # comparison transformers, capped set, partial depths: three

# ---- transformer hyperparameters (Table I) ----
MAX_LENGTH   = 128
EPOCHS       = 4
TRAIN_BS     = 16
LR           = 2e-5   # default; replaced by the sweep winner in Section 7a
RUN_LR_SWEEP = True   # Reviewer B: tuning. False = keep LR above and say "fixed in advance" in the paper
LR_GRID      = [1e-5, 2e-5, 3e-5, 5e-5]
WARMUP_FRAC  = 0.10
WEIGHT_DECAY = 0.01
FOCAL_GAMMA  = 2.0
INFER_BS     = 64     # inference batch size (report it)
N_LATENCY    = 200    # samples used for batch-size-1 latency
PROBE_C_GRID = [0.01, 0.1, 1.0, 10.0]   # linear probe: C chosen on the validation split
RUN_DEPTH_STUDY = True   # Table V (probe, Last-2, Last-4). False saves about 30 minutes and drops Table V.

# ---- classical tuning ----
CV_FOLDS       = 5
TIMING_REPEATS = 5    # classical timings = median of repeats (seed 42 only)

# ---- models ----
MODELS = {
    'MARBERTv2':    'UBC-NLP/MARBERTv2',
    # Standard AraBERT v0.2 (MSA pretraining). The Twitter variant
    # 'aubmindlab/bert-base-arabertv02-twitter' was trained at 64 tokens and its model card warns that
    # longer inputs may degrade, which would be unfair at MAX_LENGTH = 128.
    'AraBERT':      'aubmindlab/bert-base-arabertv02',
    'CAMeLBERT-DA': 'CAMeL-Lab/bert-base-arabic-camelbert-da',
    'AraBART':      'moussaKam/AraBART',
}
OTHER_MODELS = ['AraBERT', 'CAMeLBERT-DA', 'AraBART']
FP32_MODELS  = []   # add a model name here only if Section 8 reports a NaN loss for it (turns off mixed precision)
CLASSICAL_MODELS     = ['Naive Bayes', 'Logistic Regression', 'Linear SVM']
CLASSICAL_STRATEGIES = ['balanced']

GPU_HOURLY_USD = 0.35   # optional cost estimate: your own figure for a T4 hour; cite its source if you report it

if SMOKE_TEST:
    SEEDS, SEEDS_SHORT, EPOCHS, CV_FOLDS, TIMING_REPEATS, N_LATENCY = [42], [42], 1, 2, 1, 20
    LR_GRID, EXTRA_SPLIT_SEEDS = [2e-5, 5e-5], [7]
SMOKE_N = 400  # per-class sizes used in smoke-test mode

# ---- experiment plan: (model, depth, train strategy, loss, seeds) ----
REF = ('MARBERTv2', 'full', 'balanced', 'ce', SEEDS[0])   # reference run: sweep, t-SNE, extra splits
DEPTHS = ['probe', 'last2', 'last4'] if RUN_DEPTH_STUDY else []
# 1) seed-42 block: every timing row of Tables III and V. Runs first so it lands in one session.
TIMING_PLAN = [
    *[('MARBERTv2', d, 'balanced', 'ce', [SEEDS[0]]) for d in DEPTHS + ['full']],
    *[(m, 'full', 'balanced', 'ce', [SEEDS[0]]) for m in OTHER_MODELS],
]
# 2) remaining runs the reviewers asked for
MAIN_PLAN = [
    ('MARBERTv2', 'full', 'balanced', 'ce', SEEDS[1:]),                       # Reviewers A, B: seeds
    *[(m, 'full', 'balanced', 'ce', SEEDS_SHORT[1:]) for m in OTHER_MODELS],  # Reviewer C: other transformers
    ('MARBERTv2', 'full', 'capped', 'focal', SEEDS_SHORT),                    # Reviewer B: focal loss
]
# 3) extra seeds for the partial depths (nobody asked; runs last)
DEPTH_PLAN = [('MARBERTv2', d, 'balanced', 'ce', SEEDS_SHORT[1:]) for d in DEPTHS if d != 'probe']

PRED_DIR, FIG_DIR, EMB_DIR, TAB_DIR = (f'{OUT_DIR}/{d}' for d in ['predictions', 'figures', 'embeddings', 'tables'])
RESULTS_PATH = f'{OUT_DIR}/results.jsonl'
for d in [SPLIT_DIR, OUT_DIR, PRED_DIR, FIG_DIR, EMB_DIR, TAB_DIR, CKPT_DIR, os.path.dirname(IADD_PATH)]:
    os.makedirs(d, exist_ok=True)
n_ft = sum(len(s) for _, d, _, _, s in TIMING_PLAN + MAIN_PLAN + DEPTH_PLAN if d != 'probe')
print('Outputs ->', OUT_DIR)
print(f'Fine-tuning runs planned: {n_ft} + {len(LR_GRID) - 1 if RUN_LR_SWEEP else 0} sweep + {len(EXTRA_SPLIT_SEEDS)} extra-split')

## 3. Helpers (environment, results file, metrics, energy)

In [ ]:
SESSION_ID = uuid.uuid4().hex[:8]   # one id per Colab session: lets Section 8 check timing comparability

def _cpu_name():
    try:
        for line in open('/proc/cpuinfo'):
            if line.startswith('model name'):
                return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

ENV = {
    'session_id': SESSION_ID,
    'gpu_name': torch.cuda.get_device_name(0) if CUDA else 'none',
    'gpu_mem_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if CUDA else 0,
    'cpu_name': _cpu_name(),
    'n_cpus': os.cpu_count(),
    'ram_gb': round(psutil.virtual_memory().total / 2**30, 1),
    'torch': torch.__version__, 'transformers': transformers.__version__, 'sklearn': sklearn.__version__,
}
print(json.dumps(ENV, indent=1))

def load_results():
    if not os.path.exists(RESULTS_PATH) or os.path.getsize(RESULTS_PATH) == 0:
        return pd.DataFrame()
    r = pd.DataFrame([json.loads(l) for l in open(RESULTS_PATH, encoding='utf-8') if l.strip()])
    r['tag'] = r['tag'].fillna('') if 'tag' in r else ''
    return r

def done_ids():
    r = load_results()
    return set(r['run_id']) if len(r) else set()

def append_result(row):
    row = {**ENV, **row, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    with open(RESULTS_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(row, default=float) + '\n')

def make_run_id(model, depth, strategy, loss, seed, tag=''):
    # tag: '' = main run, '|lr2e-05' = learning-rate sweep, '|split7' = extra split
    return f'{model}|{depth}|{strategy}|{loss}|s{seed}{tag}'

def score(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=range(len(LABELS)), zero_division=0)
    out = {'accuracy': accuracy_score(y_true, y_pred),
           'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
           'macro_p': p.mean(), 'macro_r': r.mean()}
    for i, l in enumerate(LABELS):
        out.update({f'p_{l}': p[i], f'r_{l}': r[i], f'f1_{l}': f[i], f'support_{l}': int(s[i])})
    return out

def save_predictions(run_id, uids, y_true, y_pred):
    fn = run_id.replace('|', '__') + '.csv'
    pd.DataFrame({'uid': uids, 'y_true': y_true, 'y_pred': y_pred}).to_csv(f'{PRED_DIR}/{fn}', index=False)

def load_predictions(run_id):
    return pd.read_csv(f"{PRED_DIR}/{run_id.replace('|', '__')}.csv")

def sync():
    if CUDA:
        torch.cuda.synchronize()

class Energy:
    """CodeCarbon wrapper. Never breaks a run: if CodeCarbon fails, energy is NaN."""
    def __enter__(self):
        self.kwh = self.co2_kg = float('nan')
        try:
            from codecarbon import EmissionsTracker
            self.t = EmissionsTracker(save_to_file=False, log_level='error',
                                      measure_power_secs=1, allow_multiple_runs=True)
            self.t.start()
        except Exception as e:
            print('CodeCarbon unavailable:', e)
            self.t = None
        return self
    def __exit__(self, *exc):
        if self.t is not None:
            try:
                self.co2_kg = self.t.stop()
                self.kwh = self.t.final_emissions_data.energy_consumed
            except Exception as e:
                print('CodeCarbon stop failed:', e)
        return False

def dir_size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 2**20
    return sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(path) for f in fs) / 2**20

## 4. Data: normalization, de-duplication, fixed split

Normalization is identical to the paper (Section III-A). New: exact duplicates (after normalization) are removed and texts that IADD labels with more than one dialect are dropped, **before** splitting, so no test sentence can appear in training.

The split is built **once** and saved to `split/split.csv`. Every later session (and every model) reloads the same file.

* `test` and `val`: the same number of samples per dialect (balanced), sampled proportionally to each dialect's source corpora (AOC, DART, PADIC, SHAMI, TSAC).
* `train_pool`: everything else. Section 4b turns it into a training set.

In [ ]:
def normalize_arabic(text):
    text = re.sub('[إأآا]', 'ا', text)
    text = re.sub('ى', 'ي', text)
    text = re.sub('ؤ', 'و', text)
    text = re.sub('ئ', 'ي', text)
    text = re.sub('ة', 'ه', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)   # keep Arabic block only
    return re.sub(r'\s+', ' ', text).strip()

def _take(group, n, seed):
    """Sample n rows from one dialect, proportionally to its source corpora when possible."""
    try:
        rest, taken = train_test_split(group, test_size=n, random_state=seed, stratify=group['DataSource'])
    except ValueError:   # a source with a single row -> fall back to plain random sampling
        rest, taken = train_test_split(group, test_size=n, random_state=seed)
    return rest, taken

SPLIT_PATH = f'{SPLIT_DIR}/split.csv'
MANIFEST_PATH = f'{SPLIT_DIR}/manifest.json'

if not os.path.exists(IADD_PATH):
    print('Downloading IADD.json ...')
    urllib.request.urlretrieve(IADD_URL, IADD_PATH)
iadd_md5 = hashlib.md5(open(IADD_PATH, 'rb').read()).hexdigest()

if os.path.exists(SPLIT_PATH):
    split_df = pd.read_csv(SPLIT_PATH, encoding='utf-8')
    manifest = json.load(open(MANIFEST_PATH))
    assert manifest['iadd_md5'] == iadd_md5, 'IADD.json changed since the split was built!'
    print('Loaded existing split (built', manifest['created'], ')')
else:
    raw = pd.DataFrame(json.load(open(IADD_PATH, encoding='utf-8')))
    stats = {'raw_total': len(raw), 'raw_per_region': raw['Region'].value_counts().to_dict()}
    df = raw[raw['Region'].isin(LABELS)].dropna(subset=['Sentence']).copy()
    df['text'] = df['Sentence'].astype(str).map(normalize_arabic)
    empty = df['text'].str.len() == 0
    stats['empty_after_normalization_removed'] = df.loc[empty, 'Region'].value_counts().to_dict()
    df = df[~empty]
    stats['after_normalization'] = df['Region'].value_counts().to_dict()

    n_labels = df.groupby('text')['Region'].nunique()
    conflicting = set(n_labels[n_labels > 1].index)
    df = df[~df['text'].isin(conflicting)]
    stats['conflicting_label_texts_removed'] = len(conflicting)
    before = len(df)
    df = df.drop_duplicates('text', keep='first').reset_index(drop=True)
    stats['duplicates_removed'] = before - len(df)
    stats['after_dedup'] = df['Region'].value_counts().to_dict()
    df['uid'] = np.arange(len(df))

    n_min = df['Region'].value_counts().min()
    n_test = int(round(TEST_FRAC * n_min))
    n_val = int(round(VAL_FRAC * (n_min - n_test)))
    parts = []
    for region, g in df.groupby('Region'):
        rest, test = _take(g, n_test, SPLIT_SEED)
        rest, val = _take(rest, n_val, SPLIT_SEED)
        parts += [test.assign(split='test'), val.assign(split='val'), rest.assign(split='train_pool')]
    split_df = pd.concat(parts)[['uid', 'text', 'Region', 'DataSource', 'split']].sort_values('uid').reset_index(drop=True)
    split_df.to_csv(SPLIT_PATH, index=False, encoding='utf-8')

    manifest = {'created': time.strftime('%Y-%m-%d %H:%M'), 'iadd_md5': iadd_md5, 'split_seed': SPLIT_SEED,
                'n_test_per_class': n_test, 'n_val_per_class': n_val, **stats}
    json.dump(manifest, open(MANIFEST_PATH, 'w'), indent=1, ensure_ascii=False, default=int)
    print('Built and saved a new split.')

split_df['label'] = split_df['Region'].map(LABEL2ID)
print(json.dumps({k: v for k, v in manifest.items() if k != 'iadd_md5'}, indent=1, ensure_ascii=False))
print(pd.crosstab(split_df['Region'], split_df['split'], margins=True))

# Leakage check: no text may appear in more than one split
assert split_df['text'].is_unique, 'duplicate texts in split!'

### 4b. Training-set strategies

Validation and test sets are **identical** for every strategy, so results are directly comparable.

* `balanced`: every dialect downsampled to the smallest class (the paper's original design).
* `capped`: every dialect keeps up to `CAP_PER_CLASS` samples, so far less data is thrown away. The resulting imbalance is handled by the loss (class-weighted CE or focal loss) for transformers, and by `class_weight='balanced'` (LR, SVM) / uniform priors (NB) for classical models.

The training set is sampled once with `SPLIT_SEED`, so the five model seeds see the same data and the seed variance reflects the model only.

In [ ]:
VAL  = split_df[split_df['split'] == 'val'].reset_index(drop=True)
TEST = split_df[split_df['split'] == 'test'].reset_index(drop=True)
POOL = split_df[split_df['split'] == 'train_pool'].reset_index(drop=True)

def _per_class(df, n):
    return pd.concat([g.sample(min(len(g), n), random_state=SPLIT_SEED) for _, g in df.groupby('Region')])

if SMOKE_TEST:
    VAL, TEST = _per_class(VAL, SMOKE_N // 4).reset_index(drop=True), _per_class(TEST, SMOKE_N // 2).reset_index(drop=True)

def make_train(strategy):
    if strategy == 'balanced':
        n = POOL['Region'].value_counts().min()
    elif strategy == 'capped':
        n = CAP_PER_CLASS
    else:
        raise ValueError(strategy)
    if SMOKE_TEST:
        n = min(n, SMOKE_N if strategy == 'balanced' else 2 * SMOKE_N)
    return _per_class(POOL, n).sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

def class_weights(train):
    counts = train['label'].value_counts().sort_index().values
    return len(train) / (len(LABELS) * counts)       # 'balanced' weights, mean ~ 1

TRAIN_SETS = {s: make_train(s) for s in ['balanced', 'capped']}
for s, t in TRAIN_SETS.items():
    print(f'{s:9s} train: {len(t):6,d} ', t['Region'].value_counts().reindex(LABELS).to_dict(),
          ' class weights:', np.round(class_weights(t), 2))
print(f'val: {len(VAL):,}   test: {len(TEST):,}')

## 5. Classical baselines (TF-IDF + NB / LR / Linear SVM)

**5a. Tuning.** 5-fold stratified CV grid search, on the **training set only**, scored by macro-F1. Feature options: word uni+bigram TF-IDF with 5k/20k/50k features, or word + character (2–5, `char_wb`) TF-IDF. Best settings are saved to Drive and reused.

**5b. Final runs.** Each tuned pipeline is retrained from raw text and evaluated once on the test set. Training time **includes TF-IDF fitting** and inference time **includes vectorizing the test text**, so both are measured from raw text, exactly like the transformers. These models run on the CPU of this same GPU VM (scikit-learn does not use the GPU).

In [ ]:
FEATURE_OPTIONS = {
    'word-5k':   lambda: TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=5000),
    'word-20k':  lambda: TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=20000),
    'word-50k':  lambda: TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000),
    'word+char': lambda: FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=100000)),
    ]),
}
CLF_GRIDS = {
    'Naive Bayes':         {'clf__alpha': [0.01, 0.1, 0.5, 1.0]},
    'Logistic Regression': {'clf__C': [0.1, 1.0, 10.0]},
    'Linear SVM':          {'clf__C': [0.1, 1.0, 10.0]},
}

def make_clf(name, strategy, seed):
    weighted = strategy != 'balanced'
    if name == 'Naive Bayes':
        return MultinomialNB(fit_prior=not weighted)
    if name == 'Logistic Regression':
        return LogisticRegression(max_iter=2000, class_weight='balanced' if weighted else None, random_state=seed)
    if name == 'Linear SVM':
        return LinearSVC(max_iter=5000, class_weight='balanced' if weighted else None, random_state=seed)

def make_pipeline(name, strategy, seed, params):
    pipe = Pipeline([('feats', FEATURE_OPTIONS[params['feats']]()), ('clf', make_clf(name, strategy, seed))])
    pipe.set_params(**{k: v for k, v in params.items() if k != 'feats'})
    return pipe

TUNING_PATH = f'{OUT_DIR}/classical_best_params.json'
best_params = json.load(open(TUNING_PATH)) if os.path.exists(TUNING_PATH) else {}

for strategy in CLASSICAL_STRATEGIES:
    train = TRAIN_SETS[strategy]
    for name in CLASSICAL_MODELS:
        key = f'{name}|{strategy}'
        if key in best_params:
            print(f'[tuned] {key}: {best_params[key]}')
            continue
        feats = {k: f() for k, f in FEATURE_OPTIONS.items()}
        grid = [{'feats': [feats[k]], **CLF_GRIDS[name]} for k in feats]
        pipe = Pipeline([('feats', feats['word-5k']), ('clf', make_clf(name, strategy, SPLIT_SEED))])
        gs = GridSearchCV(pipe, grid, scoring='f1_macro', refit=False, n_jobs=-1,
                          cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SPLIT_SEED))
        t0 = time.perf_counter()
        gs.fit(train['text'].values, train['label'].values)
        best = dict(gs.best_params_)
        best['feats'] = next(k for k, v in feats.items() if v is best['feats'])
        best_params[key] = {**best, 'cv_macro_f1': round(float(gs.best_score_), 4),
                            'tuning_time_s': round(time.perf_counter() - t0, 1)}
        json.dump(best_params, open(TUNING_PATH, 'w'), indent=1)
        print(f'[tuned] {key}: {best_params[key]}  ({time.perf_counter() - t0:.0f}s)')

In [ ]:
def run_classical(name, strategy, seed, tag=''):
    run_id = make_run_id(name, 'none', strategy, 'none', seed, tag)
    train = TRAIN_SETS[strategy]
    params = {k: v for k, v in best_params[f'{name}|{strategy}'].items() if k not in ('cv_macro_f1', 'tuning_time_s')}
    X_tr, y_tr = train['text'].values, train['label'].values
    X_te = TEST['text'].values
    repeats = TIMING_REPEATS if seed == SEEDS[0] else 1

    train_times = []
    with Energy() as en:
        for _ in range(repeats):
            pipe = make_pipeline(name, strategy, seed, params)
            t0 = time.perf_counter(); pipe.fit(X_tr, y_tr); train_times.append(time.perf_counter() - t0)

    infer_times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); y_pred = pipe.predict(X_te); infer_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    for x in X_te[:N_LATENCY]:
        pipe.predict([x])
    latency_ms = (time.perf_counter() - t0) / min(N_LATENCY, len(X_te)) * 1000

    tmp = f'/content/tmp_{SESSION_ID}.joblib' if IN_COLAB else f'/tmp/tmp_{SESSION_ID}.joblib'
    joblib.dump(pipe, tmp); size_mb = dir_size_mb(tmp); os.remove(tmp)

    infer_time = float(np.median(infer_times))
    row = {'run_id': run_id, 'family': 'classical', 'model': name, 'depth': 'none', 'strategy': strategy,
           'loss': 'none', 'seed': seed, 'tag': tag, 'params': json.dumps(params),
           'n_train': len(train), 'n_val': 0, 'n_test': len(TEST),
           'train_time_s': float(np.median(train_times)), 'timing_repeats': repeats,
           'infer_time_s': infer_time, 'throughput_sps': len(X_te) / infer_time,
           'latency_ms_bs1': latency_ms, 'infer_batch_size': len(X_te),
           'model_size_mb': size_mb, 'energy_kwh_train': en.kwh / repeats, 'co2_kg_train': en.co2_kg / repeats,
           'trainable_params': None, 'total_params': None,
           **score(TEST['label'].values, y_pred)}
    save_predictions(run_id, TEST['uid'].values, TEST['label'].values, y_pred)
    append_result(row)
    print(f"{run_id:45s} acc={row['accuracy']:.4f} F1={row['macro_f1']:.4f} train={row['train_time_s']:.2f}s")

done = done_ids()
for strategy in CLASSICAL_STRATEGIES:
    for name in CLASSICAL_MODELS:
        for seed in SEEDS:
            if make_run_id(name, 'none', strategy, 'none', seed) in done:
                continue
            run_classical(name, strategy, seed)

## 6. Transformer utilities

* **Checkpoint selection** uses the validation split only (`eval_dataset=VAL`, `metric_for_best_model='macro_f1'`).
* **Depths:** `probe` = frozen backbone, logistic regression on the [CLS] embedding (C chosen on validation); `last2` / `last4` = only the top 2 / 4 encoder layers + pooler + classifier are trained; `full` = all parameters.
* **Losses:** `ce`, `weighted_ce` (class weights = n / (K·n_k)), `focal` (γ = 2, class-weighted α).
* **Timing:** training time runs from raw text (tokenization included) to the best checkpoint loaded, including the per-epoch validation passes. Inference time runs from raw test text to predictions at `INFER_BS`, after one warm-up batch.

In [ ]:
# transformers v5: AraBART's SentencePiece tokenizer no longer adds <s> ... </s>, and BART-style
# classification fails without the final </s> ("Each example must contain at least one <eos> token").
# load_tokenizer() detects this and encode() then adds the two tokens itself. Other models are untouched.
WRAP_IDS = {}   # tokenizer name -> (bos id, eos id) for tokenizers that add no special tokens

def load_tokenizer(name, config=None):
    tok = AutoTokenizer.from_pretrained(name)
    eos = getattr(config, 'eos_token_id', None)
    if config is not None and getattr(config, 'is_encoder_decoder', False) and eos is not None:
        if eos not in tok('نص تجريبي')['input_ids']:
            WRAP_IDS[tok.name_or_path] = (config.bos_token_id, eos)
            print(f'{name}: tokenizer adds no <s> ... </s>; adding ids {WRAP_IDS[tok.name_or_path]} around every text')
    return tok

def encode(tokenizer, texts, pad=False):
    """Raw texts -> model inputs (padded tensors if pad=True), truncated to MAX_LENGTH."""
    wrap = WRAP_IDS.get(tokenizer.name_or_path)
    if wrap is None:
        if pad:
            return tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        return tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
    bos = [] if wrap[0] is None else [wrap[0]]
    ids = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH - len(bos) - 1, add_special_tokens=False)['input_ids']
    ids = [bos + list(x) + [wrap[1]] for x in ids]
    enc = {'input_ids': ids, 'attention_mask': [[1] * len(x) for x in ids]}
    return tokenizer.pad(enc, return_tensors='pt') if pad else enc

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, texts, labels):
        self.enc = encode(tokenizer, texts)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = int(self.labels[i])
        return item

class DialectTrainer(Trainer):
    """Trainer with plain / class-weighted cross-entropy or focal loss."""
    def __init__(self, *args, loss_type='ce', class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_type, self.gamma = loss_type, gamma
        self.cw = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits.float()
        w = None if self.cw is None else self.cw.to(logits.device)
        if self.loss_type == 'ce':
            loss = F.cross_entropy(logits, labels)
        elif self.loss_type == 'weighted_ce':
            loss = F.cross_entropy(logits, labels, weight=w)
        elif self.loss_type == 'focal':
            logp_t = F.log_softmax(logits, dim=-1).gather(1, labels[:, None]).squeeze(1)
            focal = -((1 - logp_t.exp()) ** self.gamma) * logp_t
            loss = (focal * w[labels]).sum() / w[labels].sum() if w is not None else focal.mean()
        else:
            raise ValueError(self.loss_type)
        return (loss, outputs) if return_outputs else loss

def preprocess_logits(logits, labels):
    # BART returns (logits, encoder states); keep only the predicted class to save memory
    return (logits[0] if isinstance(logits, tuple) else logits).argmax(-1)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {'accuracy': accuracy_score(labels, preds), 'macro_f1': f1_score(labels, preds, average='macro')}

def apply_depth(model, depth):
    """Freeze everything except the top-k encoder layers, the pooler and the classifier head."""
    if depth == 'full':
        return
    k = {'last2': 2, 'last4': 4}[depth]
    base = model.base_model
    for p in base.parameters():
        p.requires_grad = False
    for layer in base.encoder.layer[-k:]:
        for p in layer.parameters():
            p.requires_grad = True
    if getattr(base, 'pooler', None) is not None:
        for p in base.pooler.parameters():
            p.requires_grad = True

def autocast():
    return torch.autocast('cuda', dtype=torch.float16) if CUDA else nullcontext()

@torch.inference_mode()
def _forward_batches(fn, tokenizer, texts, bs):
    outs = []
    for i in range(0, len(texts), bs):
        enc = encode(tokenizer, texts[i:i + bs], pad=True).to(DEVICE)
        with autocast():
            outs.append(fn(enc))
    return outs

def timed_inference(fn, tokenizer, texts, bs=INFER_BS):
    """Raw text -> outputs. Warm-up batch, then timed pass, then batch-1 latency."""
    _forward_batches(fn, tokenizer, texts[:bs], bs); sync()
    if CUDA: torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    outs = _forward_batches(fn, tokenizer, texts, bs); sync()
    dt = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float('nan')
    n_lat = min(N_LATENCY, len(texts))
    t0 = time.perf_counter()
    _forward_batches(fn, tokenizer, texts[:n_lat], 1); sync()
    latency_ms = (time.perf_counter() - t0) / n_lat * 1000
    return outs, dt, peak, latency_ms

def cls_embedding(model):
    return lambda enc: model(**enc).last_hidden_state[:, 0].float().cpu()

def logits_argmax(model):
    return lambda enc: model(**enc).logits.argmax(-1).cpu()

SIZES_PATH = f'{OUT_DIR}/model_sizes.json'
def model_size_mb(model_key, model):
    sizes = json.load(open(SIZES_PATH)) if os.path.exists(SIZES_PATH) else {}
    if model_key not in sizes:
        tmp = f'{CKPT_DIR}/size_probe'
        model.save_pretrained(tmp)
        sizes[model_key] = round(dir_size_mb(tmp), 1)
        shutil.rmtree(tmp, ignore_errors=True)
        json.dump(sizes, open(SIZES_PATH, 'w'), indent=1)
    return sizes[model_key]

def free_memory():
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

In [ ]:
def run_probe(model_key, strategy, seed):
    run_id = make_run_id(model_key, 'probe', strategy, 'ce', seed)
    set_seed(seed)
    name = MODELS[model_key]
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(DEVICE).eval()
    train = TRAIN_SETS[strategy]
    if CUDA: torch.cuda.reset_peak_memory_stats()

    with Energy() as en:
        t0 = time.perf_counter()
        X_tr = torch.cat(_forward_batches(cls_embedding(model), tok, train['text'].values, INFER_BS)).numpy()
        X_va = torch.cat(_forward_batches(cls_embedding(model), tok, VAL['text'].values, INFER_BS)).numpy()
        best = (-1, None, None)
        for C in PROBE_C_GRID:   # C chosen on the validation split
            clf = Pipeline([('scale', StandardScaler()),
                            ('lr', LogisticRegression(C=C, max_iter=3000, random_state=seed))])
            clf.fit(X_tr, train['label'].values)
            f1 = f1_score(VAL['label'].values, clf.predict(X_va), average='macro')
            if f1 > best[0]:
                best = (f1, C, clf)
        sync(); train_time = time.perf_counter() - t0
    peak_train = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float('nan')
    best_val_f1, best_C, clf = best

    fn = lambda enc: torch.from_numpy(clf.predict(model(**enc).last_hidden_state[:, 0].float().cpu().numpy()))
    outs, infer_time, peak_inf, latency = timed_inference(fn, tok, TEST['text'].values)
    y_pred = torch.cat(outs).numpy()

    total = sum(p.numel() for p in model.parameters())
    trainable = clf.named_steps['lr'].coef_.size + clf.named_steps['lr'].intercept_.size
    row = {'run_id': run_id, 'family': 'transformer', 'model': model_key, 'depth': 'probe', 'strategy': strategy,
           'loss': 'ce', 'seed': seed, 'params': json.dumps({'C': best_C}),
           'n_train': len(train), 'n_val': len(VAL), 'n_test': len(TEST),
           'best_val_macro_f1': best_val_f1, 'best_epoch': None,
           'train_time_s': train_time, 'infer_time_s': infer_time, 'throughput_sps': len(TEST) / infer_time,
           'latency_ms_bs1': latency, 'infer_batch_size': INFER_BS, 'train_batch_size': None,
           'peak_gpu_mem_train_gb': peak_train, 'peak_gpu_mem_infer_gb': peak_inf,
           'model_size_mb': model_size_mb(model_key, model), 'energy_kwh_train': en.kwh, 'co2_kg_train': en.co2_kg,
           'trainable_params': int(trainable), 'total_params': int(total + trainable),
           **score(TEST['label'].values, y_pred)}
    save_predictions(run_id, TEST['uid'].values, TEST['label'].values, y_pred)
    append_result(row)
    print(f"{run_id:45s} acc={row['accuracy']:.4f} F1={row['macro_f1']:.4f} train={train_time:.0f}s (C={best_C})")


def run_finetune(model_key, depth, strategy, loss, seed, lr=None, tag=''):
    lr = LR if lr is None else lr      # LR is set by the sweep in Section 7a
    run_id = make_run_id(model_key, depth, strategy, loss, seed, tag)
    set_seed(seed)
    name = MODELS[model_key]
    train = TRAIN_SETS[strategy]
    model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    tok = load_tokenizer(name, model.config)
    apply_depth(model, depth)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    steps = math.ceil(len(train) / TRAIN_BS) * EPOCHS
    out_dir = f"{CKPT_DIR}/{run_id.replace('|', '__')}"
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=INFER_BS,
        learning_rate=lr, weight_decay=WEIGHT_DECAY,
        warmup_steps=math.ceil(WARMUP_FRAC * steps), lr_scheduler_type='linear',
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        logging_strategy='epoch', seed=seed, data_seed=seed, fp16=CUDA and model_key not in FP32_MODELS, report_to='none')

    if CUDA: torch.cuda.reset_peak_memory_stats()
    with Energy() as en:
        t0 = time.perf_counter()
        train_ds = TextDataset(tok, train['text'].values, train['label'].values)
        val_ds = TextDataset(tok, VAL['text'].values, VAL['label'].values)   # validation only, never TEST
        trainer = DialectTrainer(
            model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
            data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
            preprocess_logits_for_metrics=preprocess_logits,
            loss_type=loss, class_weights=class_weights(train) if loss != 'ce' else None, gamma=FOCAL_GAMMA)
        trainer.train()
        sync(); train_time = time.perf_counter() - t0
    peak_train = torch.cuda.max_memory_allocated() / 2**30 if CUDA else float('nan')

    evals = [h for h in trainer.state.log_history if 'eval_macro_f1' in h]
    best_epoch = max(evals, key=lambda h: h['eval_macro_f1'])['epoch'] if evals else None
    nan_loss = any(not math.isfinite(h[k]) for h in trainer.state.log_history for k in ('loss', 'eval_loss') if k in h)

    model = trainer.model.eval()
    outs, infer_time, peak_inf, latency = timed_inference(logits_argmax(model), tok, TEST['text'].values)
    y_pred = torch.cat(outs).numpy()

    # [CLS] embeddings of the fine-tuned model for the t-SNE figure (one reference run only)
    if (model_key, depth, strategy, loss, seed) == REF and not tag.startswith('|split'):
        emb_base = f"{EMB_DIR}/marbertv2_full_cls{tag.replace('|', '_')}"
        sample = pd.concat([g.sample(min(len(g), 250), random_state=SPLIT_SEED) for _, g in TEST.groupby('Region')])
        emb = torch.cat(_forward_batches(cls_embedding(model.base_model), tok, sample['text'].values, INFER_BS)).numpy()
        np.save(emb_base + '.npy', emb)
        sample[['uid', 'label']].to_csv(emb_base + '_labels.csv', index=False)

    row = {'run_id': run_id, 'family': 'transformer', 'model': model_key, 'depth': depth, 'strategy': strategy,
           'loss': loss, 'seed': seed, 'tag': tag, 'lr': lr, 'nan_loss': bool(nan_loss),
           'params': json.dumps({'lr': lr, 'epochs': EPOCHS, 'bs': TRAIN_BS}),
           'n_train': len(train), 'n_val': len(VAL), 'n_test': len(TEST),
           'best_val_macro_f1': trainer.state.best_metric, 'best_epoch': best_epoch,
           'train_time_s': train_time, 'infer_time_s': infer_time, 'throughput_sps': len(TEST) / infer_time,
           'latency_ms_bs1': latency, 'infer_batch_size': INFER_BS, 'train_batch_size': TRAIN_BS,
           'peak_gpu_mem_train_gb': peak_train, 'peak_gpu_mem_infer_gb': peak_inf,
           'model_size_mb': model_size_mb(model_key, model), 'energy_kwh_train': en.kwh, 'co2_kg_train': en.co2_kg,
           'trainable_params': int(trainable), 'total_params': int(total),
           **score(TEST['label'].values, y_pred)}
    save_predictions(run_id, TEST['uid'].values, TEST['label'].values, y_pred)
    append_result(row)
    print(f"{run_id:45s} acc={row['accuracy']:.4f} F1={row['macro_f1']:.4f} "
          f"train={train_time:.0f}s best_epoch={best_epoch} val_F1={(trainer.state.best_metric or float('nan')):.4f}"
          + ('  <-- NaN LOSS' if nan_loss else ''))
    shutil.rmtree(out_dir, ignore_errors=True)

## 7. Run the transformers

Safe to re-run after a disconnect: finished runs (already in `results.jsonl`) are skipped. Approximate T4 times: probe ~2 min, last2/last4 ~4–5 min, full ~9–10 min on the balanced set, ~25 min on the capped set.

### 7a. Learning-rate sweep
Four learning rates on full MARBERTv2 (seed 42), chosen by **validation** macro-F1. The winning run doubles as the main seed-42 run, so the sweep costs 3 extra runs (~30 min).

In [ ]:
# ---- 7a. Learning-rate sweep (Reviewer B: tuning). Chosen on validation macro-F1 only. ----
LR_PATH = f'{OUT_DIR}/lr_sweep.json'
RETIME_FLAG = f'{OUT_DIR}/retime.flag'
lr_tag = lambda lr: f'|lr{lr:g}'
pred_file = lambda rid: f"{PRED_DIR}/{rid.replace('|', '__')}.csv"

if RUN_LR_SWEEP and not os.path.exists(LR_PATH):
    done = done_ids()
    for lr in LR_GRID:
        if make_run_id(*REF, lr_tag(lr)) not in done:
            run_finetune(*REF, lr=lr, tag=lr_tag(lr))
            free_memory()
    sw = load_results()
    sw = sw[sw['tag'].str.startswith('|lr')].sort_values(['best_val_macro_f1', 'lr'], ascending=[False, True])
    json.dump({'chosen_lr': float(sw['lr'].iloc[0]), 'grid': LR_GRID, 'selection': 'validation macro-F1, seed 42',
               'val_macro_f1': {f'{l:g}': round(float(v), 4) for l, v in zip(sw['lr'], sw['best_val_macro_f1'])}},
              open(LR_PATH, 'w'), indent=1)

if RUN_LR_SWEEP:
    info = json.load(open(LR_PATH))
    LR = info['chosen_lr']
    print('Validation macro-F1 per learning rate:', info['val_macro_f1'])
    # The winning sweep run IS the main seed-42 MARBERTv2 run: copy its row instead of training it again.
    main_id, src_id = make_run_id(*REF), make_run_id(*REF, lr_tag(LR))
    if main_id not in done_ids() and not os.path.exists(RETIME_FLAG):
        src = [r for r in map(json.loads, open(RESULTS_PATH, encoding='utf-8')) if r['run_id'] == src_id]
        with open(RESULTS_PATH, 'a', encoding='utf-8') as f:
            f.write(json.dumps({**src[-1], 'run_id': main_id, 'tag': '', 'promoted_from': src_id}) + '\n')
        shutil.copy(pred_file(src_id), pred_file(main_id))
        for ext in ['.npy', '_labels.csv']:
            s = f"{EMB_DIR}/marbertv2_full_cls{lr_tag(LR).replace('|', '_')}{ext}"
            if os.path.exists(s):
                shutil.copy(s, f'{EMB_DIR}/marbertv2_full_cls{ext}')
print(f'Learning rate used for every fine-tuning run from here on: {LR:g}')

### 7b. Seed-42 block, then the remaining seeds, other transformers and the capped set

In [ ]:
def run_plan(plan):
    done = done_ids()
    for model_key, depth, strategy, loss, seeds in plan:
        for seed in seeds:
            rid = make_run_id(model_key, depth, strategy, loss, seed)
            if rid in done:
                print('skip', rid)
                continue
            if depth == 'probe':
                run_probe(model_key, strategy, seed)
            else:
                run_finetune(model_key, depth, strategy, loss, seed)
            free_memory()

run_plan(TIMING_PLAN + MAIN_PLAN)
print('Main plan complete.')

### 7c. Extra splits
The three classical models and full MARBERTv2 (seed 42) on two more random splits of the same cleaned data.

In [ ]:
# ---- 7c. Extra splits (Reviewer A: one split). Same cleaned data, different random split. ----
# Reuses the hyperparameters tuned on the main split (say so in the paper).
def make_extra_split(seed):
    path = f'{SPLIT_DIR}/split_s{seed}.csv'
    if os.path.exists(path):
        s = pd.read_csv(path, encoding='utf-8')
    else:
        parts = []
        for region, g in split_df.groupby('Region'):
            rest, test = _take(g, manifest['n_test_per_class'], seed)
            rest, val = _take(rest, manifest['n_val_per_class'], seed)
            parts += [test.assign(split='test'), val.assign(split='val'), rest.assign(split='train_pool')]
        s = pd.concat(parts)[['uid', 'text', 'Region', 'DataSource', 'split']].sort_values('uid').reset_index(drop=True)
        s.to_csv(path, index=False, encoding='utf-8')
    s['label'] = s['Region'].map(LABEL2ID)
    return s

_main_data = (TRAIN_SETS, VAL, TEST, POOL)
try:
    for ss in EXTRA_SPLIT_SEEDS:
        s = make_extra_split(ss)
        VAL  = s[s['split'] == 'val'].reset_index(drop=True)
        TEST = s[s['split'] == 'test'].reset_index(drop=True)
        POOL = s[s['split'] == 'train_pool'].reset_index(drop=True)
        if SMOKE_TEST:
            VAL, TEST = _per_class(VAL, SMOKE_N // 4).reset_index(drop=True), _per_class(TEST, SMOKE_N // 2).reset_index(drop=True)
        TRAIN_SETS = {'balanced': make_train('balanced')}
        tag, done = f'|split{ss}', done_ids()
        print(f'--- split seed {ss}: train {len(TRAIN_SETS["balanced"]):,}  val {len(VAL):,}  test {len(TEST):,}')
        for name in CLASSICAL_MODELS:
            if make_run_id(name, 'none', 'balanced', 'none', SEEDS[0], tag) not in done:
                run_classical(name, 'balanced', SEEDS[0], tag)
        if make_run_id(*REF, tag) not in done:
            run_finetune(*REF, tag=tag)
            free_memory()
finally:
    TRAIN_SETS, VAL, TEST, POOL = _main_data
print('Extra splits complete.')

### 7d. Extra seeds for the partial depths

In [ ]:
# ---- 7d. Extra seeds for the partial depths (Table V mean ± std). Lowest priority, so it runs last. ----
run_plan(DEPTH_PLAN)
print('All runs complete.')

## 8. Analysis: tables and figures for the paper

Everything below reads `results.jsonl` and the saved predictions, so it can be run in any session (even without a GPU).

In [ ]:
R_ALL = load_results()
R_ALL['config'] = R_ALL['model'] + ' | ' + R_ALL['depth'] + ' | ' + R_ALL['strategy'] + ' | ' + R_ALL['loss']
R = R_ALL[R_ALL['tag'] == ''].copy()   # main runs only; sweep and extra-split runs are reported in the last cell

def pm(s, d=4, scale=1):
    s = s.dropna() * scale
    if len(s) == 0: return '–'
    return f'{s.mean():.{d}f} ± {s.std(ddof=1):.{d}f}' if len(s) > 1 and s.std(ddof=1) > 0 else f'{s.mean():.{d}f}'

summary = R.groupby('config').apply(lambda g: pd.Series({
    'n_runs': len(g),
    'accuracy (%)': pm(g['accuracy'], 2, 100),
    'macro F1': pm(g['macro_f1']),
    'train time (s)': pm(g['train_time_s'], 2),
    'infer time (s)': pm(g['infer_time_s'], 3),
    'throughput (samples/s)': f"{g['throughput_sps'].mean():,.0f}",
    'latency bs1 (ms)': f"{g['latency_ms_bs1'].mean():.2f}",
    'peak GPU mem train (GB)': f"{g.get('peak_gpu_mem_train_gb', pd.Series(dtype=float)).mean():.2f}",
    'energy train (Wh)': f"{g['energy_kwh_train'].mean() * 1000:.2f}",
    'trainable %': (f"{100 * g['trainable_params'].iloc[0] / g['total_params'].iloc[0]:.2f}"
                    if pd.notna(g['trainable_params'].iloc[0]) else '–'),
}), include_groups=False).sort_values('macro F1', ascending=False)
summary.to_csv(f'{TAB_DIR}/summary_all.csv')
summary

In [ ]:
# ---- Timing comparability check (Reviewer A: same hardware) ----
timing = R[(R['seed'] == SEEDS[0]) & (R['strategy'] == 'balanced')][
    ['run_id', 'session_id', 'gpu_name', 'cpu_name', 'train_time_s', 'infer_time_s']]
print(timing.to_string(index=False))
if timing['session_id'].nunique() == 1:
    print('\nOK: every seed-42 timing row comes from one session on', timing['gpu_name'].iloc[0])
elif timing['gpu_name'].nunique() == 1 and timing['cpu_name'].nunique() == 1:
    print(f"\nNOTE: the seed-42 timing rows come from {timing['session_id'].nunique()} sessions, all on the same GPU and CPU model."
          '\nThat is acceptable: in the paper write "the same Colab machine type", not "a single session".')
else:
    print('\nWARNING: the seed-42 timing rows come from different hardware.'
          '\nTo re-time them together (about 1 hour): run  drop_runs(list(timing["run_id"]))  then Runtime > Run all.')

def drop_runs(run_ids):
    """Remove runs from results.jsonl (a backup is kept) so the next Run all repeats them."""
    shutil.copy(RESULTS_PATH, RESULTS_PATH + '.bak')
    keep = [l for l in open(RESULTS_PATH, encoding='utf-8') if json.loads(l)['run_id'] not in set(run_ids)]
    open(RESULTS_PATH, 'w', encoding='utf-8').writelines(keep)
    open(RETIME_FLAG, 'w').write('re-time the main seed-42 MARBERTv2 run instead of reusing the sweep run')
    print(f'Removed {len(run_ids)} runs. Now choose Runtime > Run all.')

In [ ]:
# ---- Table III (overall) and Table V (depth frontier), balanced training set ----
def table(rows_filter, cols):
    sub = R[rows_filter]
    return sub.groupby('config').apply(lambda g: pd.Series({c: f(g) for c, f in cols.items()}), include_groups=False)

cols = {
    'Accuracy':     lambda g: pm(g['accuracy']),
    'Macro F1':     lambda g: pm(g['macro_f1']),
    'Train (s)':    lambda g: f"{g.loc[g['seed'] == SEEDS[0], 'train_time_s'].mean():.2f}",
    'Inference (s)': lambda g: f"{g.loc[g['seed'] == SEEDS[0], 'infer_time_s'].mean():.3f}",
    'Seeds':        lambda g: len(g),
}
t3 = table((R['strategy'] == 'balanced') & R['depth'].isin(['none', 'full']), cols)
t5 = table((R['model'] == 'MARBERTv2') & (R['strategy'] == 'balanced'),
           {'Params trained (%)': lambda g: f"{100 * g['trainable_params'].iloc[0] / g['total_params'].iloc[0]:.2f}", **cols})
t_imb = table((R['strategy'] == 'capped') | (R['config'] == 'MARBERTv2 | full | balanced | ce'), cols)
for nm, t in [('table3_overall', t3), ('table5_depths', t5), ('table_imbalance', t_imb)]:
    t.to_csv(f'{TAB_DIR}/{nm}.csv'); print(f'\n== {nm} ==\n', t.to_string())
print('\nTiming columns use the seed-42 run; accuracy/F1 are mean ± std over the seeds in the Seeds column.'
      '\nA value without ± is deterministic: the classical models give the same result for every seed,'
      '\nand the linear probe (frozen backbone) is run once.')

In [ ]:
# ---- Table IV: per-dialect precision / recall / F1 (mean over seeds) + support ----
rows = []
for cfg, g in R[(R['strategy'] == 'balanced') & R['depth'].isin(['none', 'full'])].groupby('config'):
    for l in LABELS:
        rows.append({'config': cfg, 'dialect': l,
                     'P': f"{g[f'p_{l}'].mean():.3f}", 'R': f"{g[f'r_{l}'].mean():.3f}",
                     'F1': pm(g[f'f1_{l}'], 3), 'support': int(g[f'support_{l}'].iloc[0])})
t4 = pd.DataFrame(rows).pivot(index='dialect', columns='config', values=['P', 'R', 'F1'])
t4.to_csv(f'{TAB_DIR}/table4_per_dialect.csv')
print('Support per class (test):', {l: int(R[f'support_{l}'].iloc[0]) for l in LABELS})
t4

In [ ]:
# ---- Significance tests and bootstrap confidence intervals ----
from scipy.stats import ttest_1samp, chi2

def mcnemar_p(rid_a, rid_b):
    """McNemar's test with continuity correction on two models' predictions for the same test set."""
    a, b = load_predictions(rid_a), load_predictions(rid_b)
    assert (a['uid'].values == b['uid'].values).all()
    ca, cb = (a['y_pred'] == a['y_true']).values, (b['y_pred'] == b['y_true']).values
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return 1.0 if n01 + n10 == 0 else float(chi2.sf((abs(n01 - n10) - 1) ** 2 / (n01 + n10), 1))

bal = R[R['strategy'] == 'balanced']
best_classical = bal[bal['family'] == 'classical'].groupby('model')['macro_f1'].mean().idxmax()
print('Best classical model (balanced):', best_classical)

MB = 'MARBERTv2 | full | balanced | ce'
comparisons = [(MB, f'{best_classical} | none | balanced | none')]
comparisons += [(MB, f'{m} | full | balanced | ce') for m in OTHER_MODELS]
comparisons += [('MARBERTv2 | full | capped | focal', MB),
                ('MARBERTv2 | last4 | balanced | ce', 'MARBERTv2 | last2 | balanced | ce'),
                (MB, 'MARBERTv2 | last4 | balanced | ce'),
                ('MARBERTv2 | last2 | balanced | ce', 'MARBERTv2 | probe | balanced | ce')]
rows = []
for cfg_a, cfg_b in comparisons:
    A, B = R[R['config'] == cfg_a].set_index('seed'), R[R['config'] == cfg_b].set_index('seed')
    common = sorted(A.index.intersection(B.index))
    if not common:
        continue
    row = {'A': cfg_a, 'B': cfg_b, 'seeds': len(common)}
    for metric, scale in [('accuracy', 100), ('macro_f1', 1)]:
        d = (A.loc[common, metric] - B.loc[common, metric]).astype(float)
        row[f'mean diff {metric}' + (' (pts)' if scale == 100 else '')] = round(scale * d.mean(), 4)
        row[f't-test p ({metric})'] = round(float(ttest_1samp(d, 0).pvalue), 4) if len(d) > 1 and d.std() > 0 else None
    ps = [mcnemar_p(A.loc[s, 'run_id'], B.loc[s, 'run_id']) for s in common]
    row['McNemar p, largest over seeds'] = f'{max(ps):.2e}'
    row['seeds with McNemar p < 0.05'] = f'{sum(p < 0.05 for p in ps)}/{len(ps)}'
    rows.append(row)
sig = pd.DataFrame(rows)
sig.to_csv(f'{TAB_DIR}/table_significance.csv', index=False)
print('\nA minus B. The t-test is paired across seeds (low power with 3 seeds); McNemar is per seed on the test set.')
print(sig.to_string(index=False))

# Bootstrap 95% CIs of test accuracy and macro-F1 (seed-42 runs, same 1,000 resamples for every model)
N_BOOT, K = 1000, len(LABELS)
boot_runs = R[(R['seed'] == SEEDS[0]) & R['depth'].isin(['none', 'full'])].sort_values('macro_f1', ascending=False)
IDX, rows = None, []
for _, r in boot_runs.iterrows():
    p = load_predictions(r['run_id']); yt, yp = p['y_true'].values, p['y_pred'].values
    if IDX is None:
        IDX = np.random.default_rng(0).integers(0, len(yt), (N_BOOT, len(yt)))
    acc = (yt == yp)[IDX].mean(1)
    f1 = []
    for i in IDX:
        cm = np.bincount(yt[i] * K + yp[i], minlength=K * K).reshape(K, K)
        f1.append(np.mean(2 * np.diag(cm) / np.maximum(cm.sum(0) + cm.sum(1), 1)))
    rows.append({'config': r['config'],
                 'accuracy (%)': f"{100 * r['accuracy']:.2f}",
                 'accuracy 95% CI': f'[{100 * np.percentile(acc, 2.5):.2f}, {100 * np.percentile(acc, 97.5):.2f}]',
                 'macro F1': f"{r['macro_f1']:.4f}",
                 'macro F1 95% CI': f'[{np.percentile(f1, 2.5):.4f}, {np.percentile(f1, 97.5):.4f}]'})
ci = pd.DataFrame(rows)
ci.to_csv(f'{TAB_DIR}/table_bootstrap_ci.csv', index=False)
print(f'\nBootstrap 95% CIs (seed {SEEDS[0]}, {N_BOOT} resamples of the test set):')
print(ci.to_string(index=False))

In [ ]:
# ---- Figure: 2x2 confusion matrices (seed 42, balanced) ----
panel = [(m, make_run_id(m, 'none', 'balanced', 'none', SEEDS[0])) for m in CLASSICAL_MODELS] + \
        [('MARBERTv2', make_run_id('MARBERTv2', 'full', 'balanced', 'ce', SEEDS[0]))]
panel = [(t, r) for t, r in panel if r in set(R['run_id'])]
cms = [confusion_matrix(load_predictions(r)['y_true'], load_predictions(r)['y_pred'], labels=range(len(LABELS))) for _, r in panel]
vmax = max(cm.max() for cm in cms)
fig, axes = plt.subplots(2, 2, figsize=(7, 6.4))
for ax, (title, _), cm in zip(axes.flat, panel, cms):
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=vmax)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9,
                    color='white' if cm[i, j] > vmax / 2 else 'black')
    ax.set_xticks(range(len(LABELS)), LABELS, fontsize=8); ax.set_yticks(range(len(LABELS)), LABELS, fontsize=8)
    ax.set_title(title, fontsize=10)
for ax in axes[1]: ax.set_xlabel('Predicted', fontsize=9)
for ax in axes[:, 0]: ax.set_ylabel('Actual', fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.7)
plt.savefig(f'{FIG_DIR}/fig2_confusion_matrices.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_DIR}/fig2_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# Confusion matrices for every model (seed-42 runs), as numbers
all_cm = []
for _, r in R[(R['seed'] == SEEDS[0]) & R['depth'].isin(['none', 'full'])].iterrows():
    p = load_predictions(r['run_id'])
    cm = pd.DataFrame(confusion_matrix(p['y_true'], p['y_pred'], labels=range(len(LABELS))), LABELS, LABELS)
    all_cm.append(cm.rename_axis('actual').reset_index().assign(config=r['config']))
pd.concat(all_cm).to_csv(f'{TAB_DIR}/confusion_matrices_all.csv', index=False)
print(f'Caption: confusion matrices on the test set (n = {len(TEST):,}), from the seed-{SEEDS[0]} run of each model.')

In [ ]:
# ---- Accuracy by source corpus, and on AOC texts only (dataset-makeup point, Reviewer A) ----
src = TEST[['uid', 'Region', 'DataSource']]
ref_runs = R[(R['seed'] == SEEDS[0]) & (R['strategy'] == 'balanced') & R['depth'].isin(['none', 'full'])]
rows, aoc_rows = [], []
for _, r in ref_runs.iterrows():
    p = load_predictions(r['run_id']).merge(src, on='uid')
    p['ok'] = p['y_true'] == p['y_pred']
    for (reg, ds), g in p.groupby(['Region', 'DataSource']):
        rows.append({'model': r['model'], 'dialect': reg, 'source': ds, 'n': len(g), 'accuracy': round(g['ok'].mean(), 3)})
    a = p[p['DataSource'] == 'AOC']
    aoc_rows.append({'model': r['model'], 'accuracy, all test (%)': round(100 * p['ok'].mean(), 2),
                     'accuracy, AOC only (%)': round(100 * a['ok'].mean(), 2) if len(a) else None, 'n AOC': len(a)})
by_source = pd.DataFrame(rows).pivot_table(index=['dialect', 'source', 'n'], columns='model', values='accuracy')
by_source.to_csv(f'{TAB_DIR}/accuracy_by_source.csv')
aoc_only = pd.DataFrame(aoc_rows)
aoc_only.to_csv(f'{TAB_DIR}/table_aoc_only.csv', index=False)
print('AOC test texts per dialect:', src[src['DataSource'] == 'AOC']['Region'].value_counts().to_dict())
print('(AOC-only accuracy covers only the dialects listed above; say so in the paper.)\n')
print(aoc_only.to_string(index=False))
by_source

In [ ]:
# ---- Figure: Jaccard similarity of each dialect's top-300 TF-IDF terms (balanced training set) ----
tr = TRAIN_SETS['balanced']
vec = TfidfVectorizer(analyzer='word', min_df=2, sublinear_tf=True)
X = vec.fit_transform(tr['text'])
vocab = np.array(vec.get_feature_names_out())
top = {l: set(vocab[np.asarray(X[(tr['Region'] == l).values].mean(axis=0)).ravel().argsort()[-300:]]) for l in LABELS}
J = pd.DataFrame([[len(top[a] & top[b]) / len(top[a] | top[b]) for b in LABELS] for a in LABELS], LABELS, LABELS)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(np.where(np.eye(len(LABELS)), np.nan, J), cmap='viridis', vmin=0, vmax=1)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j: ax.text(j, i, f'{J.iloc[i, j]:.3f}', ha='center', va='center', color='white', fontsize=8)
ax.set_xticks(range(len(LABELS)), LABELS); ax.set_yticks(range(len(LABELS)), LABELS)
fig.colorbar(im, ax=ax, label='Jaccard similarity')
plt.savefig(f'{FIG_DIR}/fig3_jaccard.pdf', bbox_inches='tight'); plt.show()
J.round(3)

In [ ]:
# ---- Figure: t-SNE of fine-tuned MARBERTv2 [CLS] embeddings (test sample, seed 42) ----
from sklearn.manifold import TSNE
emb_path = f'{EMB_DIR}/marbertv2_full_cls.npy'
if os.path.exists(emb_path):
    E = np.load(emb_path); lab = pd.read_csv(f'{EMB_DIR}/marbertv2_full_cls_labels.csv')['label'].values
    Z = TSNE(n_components=2, perplexity=30, init='pca', random_state=SPLIT_SEED).fit_transform(E)
    fig, ax = plt.subplots(figsize=(5, 4))
    for i, l in enumerate(LABELS):
        ax.scatter(*Z[lab == i].T, s=8, alpha=0.7, label=l, marker='osd^'[i])
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.legend(title='Dialect', fontsize=8)
    plt.savefig(f'{FIG_DIR}/fig4_tsne.pdf', bbox_inches='tight'); plt.show()
else:
    print('Embeddings not found: run the MARBERTv2 full / balanced / seed-42 run first.')

In [ ]:
# ---- Table II: environment and resources (from the seed-42 timing session) ----
ref = R[(R['seed'] == SEEDS[0]) & (R['strategy'] == 'balanced')]
mb = ref[(ref['model'] == 'MARBERTv2') & (ref['depth'] == 'full')]
env_row = (mb if len(mb) else ref).iloc[0]
sizes = json.load(open(SIZES_PATH)) if os.path.exists(SIZES_PATH) else {}
gpu_hours = mb['train_time_s'].sum() / 3600 if len(mb) else float('nan')
resources = {
    'Platform': 'Google Colab',
    'GPU': f"{env_row['gpu_name']} ({env_row['gpu_mem_gb']} GB)",
    'CPU / RAM': f"{env_row['cpu_name']}, {env_row['n_cpus']} vCPU / {env_row['ram_gb']} GB",
    'Software': f"PyTorch {env_row['torch']}, Transformers {env_row['transformers']}, scikit-learn {env_row['sklearn']}",
    'Train / inference batch size': f'{TRAIN_BS} / {INFER_BS}',
    'Peak GPU memory, MARBERTv2 full (train / infer)':
        f"{mb['peak_gpu_mem_train_gb'].mean():.2f} / {mb['peak_gpu_mem_infer_gb'].mean():.2f} GB" if len(mb) else '–',
    'Model size on disk': ', '.join(f'{k}: {v:.0f} MB' for k, v in sizes.items()) + ' | classical: ' +
        ', '.join(f"{m}: {ref.loc[ref['model'] == m, 'model_size_mb'].mean():.1f} MB" for m in CLASSICAL_MODELS),
    'Throughput, MARBERTv2 full': f"{mb['throughput_sps'].mean():,.0f} samples/s" if len(mb) else '–',
    'Energy, MARBERTv2 full fine-tune': f"{1000 * mb['energy_kwh_train'].mean():.1f} Wh" if len(mb) else '–',
    'Cost estimate, MARBERTv2 full fine-tune': f'≈ ${gpu_hours * GPU_HOURLY_USD:.3f} at ${GPU_HOURLY_USD}/GPU-hour',
}
pd.Series(resources).to_csv(f'{TAB_DIR}/table2_resources.csv')
print('\n'.join(f'{k}: {v}' for k, v in resources.items()))

# Per-model resources (seed-42 runs, balanced training set): the rows of Table II
res = R[(R['seed'] == SEEDS[0]) & (R['strategy'] == 'balanced') & R['depth'].isin(['none', 'full'])].copy()
for c in ['peak_gpu_mem_train_gb', 'peak_gpu_mem_infer_gb']:
    if c not in res: res[c] = np.nan
per_model = pd.DataFrame({
    'Model': res['model'].values,
    'Device': np.where(res['family'] == 'classical', 'CPU', res['gpu_name']),
    'Params (M)': (pd.to_numeric(res['total_params']) / 1e6).round(1).values,
    'Size on disk (MB)': res['model_size_mb'].round(1).values,
    'Peak GPU mem train / infer (GB)': [f'{a:.2f} / {b:.2f}' if pd.notna(a) else '–'
                                        for a, b in zip(res['peak_gpu_mem_train_gb'], res['peak_gpu_mem_infer_gb'])],
    'Train (s)': res['train_time_s'].round(2).values,
    'Inference (s)': res['infer_time_s'].round(3).values,
    'Throughput (texts/s)': res['throughput_sps'].round(0).values,
    'Latency bs1 (ms)': res['latency_ms_bs1'].round(2).values,
    'Energy per training run (Wh, CodeCarbon estimate)': (1000 * res['energy_kwh_train']).round(3).values,
})
per_model.to_csv(f'{TAB_DIR}/table2_per_model_resources.csv', index=False)
per_model

In [ ]:
# ---- Numbers to update in the paper text (abstract, results, conclusion) ----
def mean_of(cfg, col='accuracy'):
    v = R.loc[R['config'] == cfg, col]
    return v.mean() if len(v) else float('nan')

mb_full = 'MARBERTv2 | full | balanced | ce'
bc = f'{best_classical} | none | balanced | none'
nums = {
    'MARBERTv2 accuracy (%)': 100 * mean_of(mb_full),
    'MARBERTv2 macro F1': mean_of(mb_full, 'macro_f1'),
    f'{best_classical} accuracy (%)': 100 * mean_of(bc),
    f'{best_classical} macro F1': mean_of(bc, 'macro_f1'),
    'Gap (pts)': 100 * (mean_of(mb_full) - mean_of(bc)),
    **{f'{d} accuracy (%)': 100 * mean_of(f'MARBERTv2 | {d} | balanced | ce') for d in ['probe', 'last2', 'last4']},
    'Share of gap recovered by probe (%)': 100 * (mean_of('MARBERTv2 | probe | balanced | ce') - mean_of(bc)) /
                                            (mean_of(mb_full) - mean_of(bc)),
    'Share of gap recovered by last2 (%)': 100 * (mean_of('MARBERTv2 | last2 | balanced | ce') - mean_of(bc)) /
                                            (mean_of(mb_full) - mean_of(bc)),
}
pd.Series(nums).round(2).to_frame('value')

In [ ]:
# ---- Learning-rate sweep, extra splits, sanity checks, and the counts the paper text needs ----
sw = R_ALL[R_ALL['tag'].str.startswith('|lr')]
if len(sw):
    t = sw[['lr', 'best_val_macro_f1']].sort_values('lr').rename(columns={'best_val_macro_f1': 'validation macro F1'})
    t['lr'] = t['lr'].map('{:g}'.format)
    t.to_csv(f'{TAB_DIR}/table_lr_sweep.csv', index=False)
    print(f'== Learning-rate sweep (MARBERTv2 full, seed {SEEDS[0]}; chosen on validation only) ==')
    print(t.to_string(index=False), f'\nChosen: {LR:g}. The other transformers use this value too (say so in the paper).\n')

sp = R_ALL[(R_ALL['seed'] == SEEDS[0]) & (R_ALL['strategy'] == 'balanced') & R_ALL['depth'].isin(['none', 'full']) &
           (R_ALL['tag'].str.startswith('|split') | (R_ALL['tag'] == '')) &
           R_ALL['model'].isin(CLASSICAL_MODELS + ['MARBERTv2'])].copy()
if sp['tag'].nunique() > 1:
    sp['split seed'] = sp['tag'].str.replace('|split', '', regex=False).replace('', str(SPLIT_SEED))
    detail = sp.pivot_table(index='model', columns='split seed', values=['accuracy', 'macro_f1'])
    detail.to_csv(f'{TAB_DIR}/table_splits_detail.csv')
    splits = pd.DataFrame({'Accuracy (%)': sp.groupby('model')['accuracy'].apply(lambda s: pm(s, 2, 100)),
                           'Macro F1': sp.groupby('model')['macro_f1'].apply(pm),
                           'Splits': sp.groupby('model').size()})
    splits.to_csv(f'{TAB_DIR}/table_splits.csv')
    print(f'== Mean ± std over {sp["tag"].nunique()} splits (seed {SEEDS[0]}; hyperparameters from the main split) ==')
    print(splits.to_string(), '\n')

bad = R_ALL[R_ALL.get('nan_loss', pd.Series(False, index=R_ALL.index)).fillna(False).astype(bool)]
print('Runs with a NaN or infinite loss:', list(bad['run_id']) if len(bad) else 'none')
if len(bad):
    print('  Fix: add that model to FP32_MODELS in Section 2, run drop_runs([...those run ids...]), then Run all.')

planned = {make_run_id(m, d, s, l, seed) for m, d, s, l, seeds in TIMING_PLAN + MAIN_PLAN + DEPTH_PLAN for seed in seeds}
missing = sorted(planned - set(R_ALL['run_id']))
print('Planned runs still missing:', missing if missing else 'none')

print('\n== Counts for the paper text ==')
print(json.dumps({k: v for k, v in manifest.items() if k != 'iadd_md5'}, indent=1, ensure_ascii=False))
for s_, t_ in TRAIN_SETS.items():
    print(f'{s_} training set: {len(t_):,}', t_['Region'].value_counts().reindex(LABELS).to_dict())
print(f'validation: {len(VAL):,}   test: {len(TEST):,}')
print('Selected classical settings:', json.dumps(best_params, indent=1))
print('Seeds:', SEEDS, '| 3-seed configurations use', SEEDS_SHORT)